In [1]:
"""Scripts for analyzing of phantom outputs.

This script writes hdf5 files for each dump (and one hdf5 file synthsizing all outputs)
    to plot photosphere size vs time or orbital separation.
It does so by plotting photosphere intersection with traced rays originating from the primary star
    and shooting along the axes of the coordination frame.

"""

'Scripts for analyzing of phantom outputs.\n\nThis script writes hdf5 files for each dump (and one hdf5 file synthsizing all outputs)\n    to plot photosphere size vs time or orbital separation.\nIt does so by plotting photosphere intersection with traced rays originating from the primary star\n    and shooting along the axes of the coordination frame.\n\n'

## Imports & Settings

In [2]:
#%matplotlib inline
import math
import numpy as np
from numpy import pi
#import pandas
from astropy import units
from astropy import constants as const
import matplotlib.pyplot as plt
import matplotlib as mpl
#from moviepy.editor import ImageSequenceClip

In [3]:
# import modules listed in ./lib/

import clmuphantomlib as mupl
from clmuphantomlib import get_col_kernel_funcs
from clmuphantomlib.io import json_load, json_dump, hdf5_load, hdf5_dump
from clmuphantomlib.settings import DEFAULT_SETTINGS as settings
from clmuphantomlib.log import error, warn, note, debug_info
from clmuphantomlib.log import is_verbose, say
from clmuphantomlib.units_util import set_as_quantity

    ## import modules in arbitrary directory
    
    #import sys
    
    ## path to my python module lib directory
    ## *** CHECK THIS! *** #
    #SRC_LIB_PATH = sys.path[0] + '/lib'
    
    #if SRC_LIB_PATH not in sys.path:
    #    sys.path.append(SRC_LIB_PATH)
    ##print(*sys.path, sep='\n')    # debug
    #print(
    #    "\n*   Please Make sure my module files are located in this directory (or change the SRC_LIB_PATH variable):",
    #    f"\n{SRC_LIB_PATH = }\n"
    #)

In [4]:
# parallels & optimizations


#import os
## Fixing stupid numba killing kernel
## See here https://github.com/numba/numba/issues/3016
#os.environ['NUMBA_DISABLE_INTEL_SVML']  = '1'
#from numba import njit, prange


from multiprocessing import cpu_count, Pool #Process, Queue
NPROCESSES = 1 if cpu_count() is None else max(cpu_count(), 1)


In [5]:
# settings
#
#   imported from script_input.py file

from script_PhLocAxes__input import interm_dir, verbose, PHOTOSPHERE_TAU, JOB_PROFILES
from _sharedFuncs import mpdf_read


# set metadata
with open("_metadata__input.json", 'r') as f:
    metadata = mupl.json_load(f)
metadata['Title'] = "Getting photosphere size on x, y, z axes"
metadata['Description'] = f"""Tracing 6 rays on +x, -x, +y, -y, +z, -z directon and get photosphere size, h, rho, u, T from them."""


plt.rcParams.update({'font.size': 20})


# print debug info
if __name__ == '__main__' and is_verbose(verbose, 'note'):
    # remember to check if name is '__main__' if you wanna say anything
    #    so when you do multiprocessing the program doesn't freak out
    say('note', "script", verbose, f"Will use {NPROCESSES} processes for parallelization")
    

*   Note   :    script:
	Will use 32 processes for parallelization


In [6]:
#  import (my libs)
from clmuphantomlib.log import say, is_verbose
from clmuphantomlib.geometry import get_dist2_between_2pt, get_closest_pt_on_line, get_dist2_from_pt_to_line_nb, get_ray_unit_vec, get_rays_unit_vec
from clmuphantomlib.sph_interp import get_sph_interp, get_h_from_rho, get_no_neigh
from clmuphantomlib.units_util import set_as_quantity, set_as_quantity_temperature, get_units_field_name
from clmuphantomlib.eos.base import EoS_Base
#  import (general)
import numpy as np
from numpy import typing as npt
import numba
from numba import jit, prange
import sarracen
import itertools

In [7]:
from script_PhLocAxes__defs import write_ph_loc_axes

# Analysis

## Photosphere size vs time

## Main

In [9]:
do_debug = True
if do_debug and __name__ == '__main__':
    CALC_PARAMS = ['loc', 'R1', 'rho', 'h', 'T', 'kappa', 'srcfunc', 'Skapparho']
    # CALC_PARAMS = ['loc', 'R1', 'rho', 'h', 'T', 'kappa', 'kappaDust', 'srcfunc', 'Skapparho']
    from script_PhLocAxes__input import JOB_PROFILES_DICT
    JOB_PROFILES = [JOB_PROFILES_DICT[key] for key in (
        # '2ma-star--mfrac-p2-08', '2ma-star--mfrac-p2-10', '2ma-star--mfrac-p2-12','2m-star--np1376256',
        '2ma-lumo--mfrac-p2-08', '2ma-lumo--mfrac-p2-10',
    )]
    for job_profile in JOB_PROFILES:
        inds = job_profile['file_indexes']
        # job_profile['file_indexes'] = np.asarray([0, 400, 800, 960, 1200])
        # #inds[inds % 100 == 0]
        # #(0, 1200, 4800, 8000, 17600)
        # #(0, 400, 800, 1200, 1600, 2000, 2400, 2800, 3200, 4800, 6400, 8000, 15600, 17600)
        print(job_profile['nickname'], job_profile['file_indexes'])
    # NPROCESSES = 1
    verbose=6

2ma-lumo--mfrac-p2-08 [   0  100  200  300  400  500  600  700  800  900 1000 1100 1200 1300
 1400 1500 1600 1700 1800 1900 2000 2100 2200 2300 2400 2500 2600 2700
 2800 2900 3000 3100 3200 3300 3400 3500 3600 3700 3800 3900 4000]
2ma-lumo--mfrac-p2-10 [  0  50 100 150 200 250 300 350 400 450 500 550]


In [11]:
# init rays directions
rays_dir_def = {
    # legend: ray direction name
    '+x'  : [ 1., 0., 0.],
    '+y'  : [ 0., 1., 0.],
    '+z'  : [ 0., 0., 1.],
    '-x'  : [-1., 0., 0.],
    '-y'  : [ 0.,-1., 0.],
    '-z'  : [ 0., 0.,-1.],
}


In [10]:
# main process

if __name__ == '__main__':
    
    
    # get ph loc for each dump file
    args = []
    eoses = []
    eoses_opacity = []
    for job_profile in JOB_PROFILES:
    
        file_indexes = job_profile['file_indexes']
        # if job_profile['job_name'] == "2ma-lumo--mfrac-p2-08": file_indexes = [3900, 4000]
        #job_name     = job_profile['job_name']
        eos          = mupl.get_eos(job_profile['ieos'], job_profile['params'], settings)
        eos_opacity  = mupl.eos.mesa.EoS_MESA_opacity(job_profile['params'], settings)
    
        
        if NPROCESSES <= 1:
            
            # single process
    
            photosphere_pars = write_ph_loc_axes(
                job_profile = job_profile, file_indexes = file_indexes, rays_dir_def = rays_dir_def,
                eoses = (eos, eos_opacity), calc_params=CALC_PARAMS, photosphere_tau = PHOTOSPHERE_TAU, verbose = verbose,
            )
            
        else:
            
            # multi-process

            eoses.append(eos)
            eoses_opacity.append(eos_opacity)
            for file_index in file_indexes:
                args.append((
                    job_profile,
                    [file_index],
                    rays_dir_def,
                    (eoses[-1], eoses_opacity[-1]),
                    CALC_PARAMS,
                    PHOTOSPHERE_TAU,
                    0,
                ))

    if NPROCESSES > 1:
        print("Starting processes...")
        with Pool(processes=NPROCESSES) as pool:
            pool.starmap(write_ph_loc_axes, args)
    
    


Starting processes...
*   Note   :    starmapstar() ==> write_ph_loc_axes() ==> hdf5_dump():
	Writing to ../interm/2ma-lumo--mfrac-p2-08_02300.photospherePars.xyz.hdf5  (will OVERWRITE if file already exist.; compress=False)
*   Note   :    starmapstar() ==> write_ph_loc_axes() ==> hdf5_dump():
	Writing to ../interm/2ma-lumo--mfrac-p2-10_00450.photospherePars.xyz.hdf5  (will OVERWRITE if file already exist.; compress=False)
*   Note   :    starmapstar() ==> write_ph_loc_axes() ==> hdf5_dump():
	Writing to ../interm/2ma-lumo--mfrac-p2-08_00200.photospherePars.xyz.hdf5  (will OVERWRITE if file already exist.; compress=False)
*   Note   :    starmapstar() ==> write_ph_loc_axes() ==> hdf5_dump():
	Writing to ../interm/2ma-lumo--mfrac-p2-08_03300.photospherePars.xyz.hdf5  (will OVERWRITE if file already exist.; compress=False)
*   Note   :    starmapstar() ==> write_ph_loc_axes() ==> hdf5_dump():
	Writing to ../interm/2ma-lumo--mfrac-p2-08_00100.photospherePars.xyz.hdf5  (will OVERWRITE if 

In [12]:
if __name__ == '__main__':
    
    # syntheize the files into one big file
    
    for job_profile in JOB_PROFILES:
    
        job_name     = job_profile['job_name']
        file_indexes = job_profile['file_indexes']
    
    
        # init
        photosphere_pars_all = { # [legend][par_name][time]
            'time_yr': [],
            'orbsep_Rsun': [],
            'data': {},
            'rays_dir': rays_dir_def,
            'rays': {},
        }  
        for key in rays_dir_def.keys():
            photosphere_pars_all['data'][key] = {
                'size': [],
                'R1'  : [],
                'rho' : [],
                # 'u'   : [],
                'h'   : [],
                'T'   : [],
                'R1_on_ray' : [],
                'tau_on_ray': [],
                'rho_on_ray': [],
                # 'u_on_ray'  : [],
                'T_on_ray'  : [],
            }
            photosphere_pars_all['rays'][key] = []
    
        
        # fetch
        for file_index in file_indexes:
                
            if verbose: print(f"\n\nLoading {f.name}... ", end='')
            
            photosphere_pars = hdf5_load(f"{interm_dir}{job_profile['nickname']}_{file_index:05}.photospherePars.xyz.hdf5")
            for it in ['time_yr', 'orbsep_Rsun']:
                photosphere_pars_all[it].append(photosphere_pars[it])
            for key in rays_dir_def.keys():
                for it in photosphere_pars_all['data'][key].keys():
                    obj = photosphere_pars['data'][key][it]
                    if isinstance(obj, np.ndarray):
                        obj = obj.tolist()
                    photosphere_pars_all['data'][key][it].append(np.asanyarray(obj))
                photosphere_pars_all['rays'][key].append(photosphere_pars['rays'][key]) 

            if verbose: print(f"Done.\n")
        
        # write
        hdf5_dump(
            photosphere_pars_all,
            f"{interm_dir}{job_profile['nickname']}.photospherePars.xyz.hdf5",
            metadata=metadata)


    print("\n\n\n*** All Done. ***\n\n\n")



Loading _metadata__input.json... *   Note   :    run_code() ==> <module>() ==> hdf5_load():
	Reading from ../interm/2ma-lumo--mfrac-p2-08_00000.photospherePars.xyz.hdf5  (compress=False)
Done.



Loading _metadata__input.json... *   Note   :    run_code() ==> <module>() ==> hdf5_load():
	Reading from ../interm/2ma-lumo--mfrac-p2-08_00100.photospherePars.xyz.hdf5  (compress=False)
Done.



Loading _metadata__input.json... *   Note   :    run_code() ==> <module>() ==> hdf5_load():
	Reading from ../interm/2ma-lumo--mfrac-p2-08_00200.photospherePars.xyz.hdf5  (compress=False)
Done.



Loading _metadata__input.json... *   Note   :    run_code() ==> <module>() ==> hdf5_load():
	Reading from ../interm/2ma-lumo--mfrac-p2-08_00300.photospherePars.xyz.hdf5  (compress=False)
Done.



Loading _metadata__input.json... *   Note   :    run_code() ==> <module>() ==> hdf5_load():
	Reading from ../interm/2ma-lumo--mfrac-p2-08_00400.photospherePars.xyz.hdf5  (compress=False)
Done.



Loading _metadata_